## Creates patterns and semantic_annotations tables

### all annotated verb patterns

In [1]:
import sqlite3
import pandas as pd
from estnltk import Text
import sys
sys.path.append("..")
from common_display import display_db_table 

## Configuration

In [2]:
DB_DIR = "../example_data"
SOURCE_DIR = "../source_data"

# annoteeritud andmed
ANNOTATION_FILE = f"{SOURCE_DIR}/verb_patterns.csv"

VERB_PATTERN_DB = f"{DB_DIR}/verb_patterns.db"

# patterns lõpptabel
PATTERN_TABLE_NAME = "patterns" 
# semnatiliste annotatsioonide tabel
SEMANTIC_ANNOTATIONS_TABLE = "semantic_annotations"

## Create verb_patterns database

In [3]:
con = sqlite3.connect(VERB_PATTERN_DB)
cur = con.cursor()

## Workflow

### Read in and process verb annotations

In [4]:
df = pd.read_csv(ANNOTATION_FILE, sep=";", encoding="utf-8")
df = df.fillna('')
df

,pat_id,isik,koht,muu,deprel,phrase_case,pattern,verb_word,verb_compound,phrase_nr
0,1,vahel,vahel,mitte kunagi,obl,abl,saama kellelt/millelt,saama,,1
1,2,vahel,vahel,mitte kunagi,obl,abl,tulema kellelt/millelt,tulema,,1
2,3,alati,mitte kunagi,mitte kunagi,obl,abl,küsima kellelt/millelt,küsima,,1
3,4,alati,mitte kunagi,mitte kunagi,obl,abl,nõudma kellelt/millelt,nõudma,,1
4,5,vahel,vahel,mitte kunagi,obl,abl,võtma kellelt/millelt,võtma,,1
...,...,...,...,...,...,...,...,...,...,...
10601,10602,mitte kunagi,alati,muu,obl,in,musitseerima kelles/milles,musitseerima,,1
10602,10603,mitte kunagi,mitte kunagi,muu,obl,in,kõigutama kelles/milles,kõigutama,,1
10603,10604,mitte kunagi,alati,muu,obl,in,kätlema kelles/milles,kätlema,,1
10604,10605,mitte kunagi,alati,mitte kunagi,obl,in,kõmmutama kelles/milles,kõmmutama,,1


### Create patterns table in database

In [5]:
cur.execute("""DROP TABLE IF EXISTS {name}""".format(name=PATTERN_TABLE_NAME))

cur.execute(
    """CREATE TABLE {tablename}
    (pat_id INTEGER PRIMARY KEY, pattern TEXT, phrase_case TEXT, verb_word TEXT, verb_compound TEXT, phrase_nr INT, deprel TEXT)
    """.format(tablename=PATTERN_TABLE_NAME)
)


insert = f"INSERT INTO {PATTERN_TABLE_NAME} "
for idx, row in df.iterrows():
    cur.execute(insert + """
    (pat_id, pattern, phrase_case, verb_word, verb_compound, phrase_nr, deprel) 
    VALUES (?, ?, ?, ?, ?, ?, ?);""", 
    (row["pat_id"], row['pattern'], row['phrase_case'], row['verb_word'], row['verb_compound'], row['phrase_nr'], row['deprel']))
    
    con.commit()

In [6]:
query = """SELECT count(*) as cnt from {new_table}""".format(new_table = PATTERN_TABLE_NAME)
r1 = pd.read_sql_query(query, con)
assert r1["cnt"][0] == 10606, "mustrite tabelis ei ole eelduslik arv ridu"

display_db_table(con, PATTERN_TABLE_NAME)

,pat_id,pattern,phrase_case,verb_word,verb_compound,phrase_nr,deprel
0,1,saama kellelt/millelt,abl,saama,,1,obl
1,2,tulema kellelt/millelt,abl,tulema,,1,obl
2,3,küsima kellelt/millelt,abl,küsima,,1,obl
3,4,nõudma kellelt/millelt,abl,nõudma,,1,obl
4,5,võtma kellelt/millelt,abl,võtma,,1,obl


### Create semantic_annotations table in database

In [7]:
cur.execute("""DROP TABLE IF EXISTS {name}""".format(name=SEMANTIC_ANNOTATIONS_TABLE))

cur.execute(
    """CREATE TABLE {tablename}
    (pattern_id INT,phrase_nr INT, semantic_role TEXT, certainty TEXT)
    """.format(tablename=SEMANTIC_ANNOTATIONS_TABLE)
)


for idx, row in df.iterrows():

    pat_id = row["pat_id"]
    phrase_nr = row["phrase_nr"]
    
    for role in ["isik", "koht", "muu"]:
        certain = row[role]

        insert = f"INSERT INTO {SEMANTIC_ANNOTATIONS_TABLE} "
        cur.execute(insert + """
            (pattern_id, phrase_nr, semantic_role, certainty) 
            VALUES (?, ?, ?, ?);""", 
            (pat_id, phrase_nr, role, certain)
        )
    
        con.commit()

In [8]:
query = """SELECT count(*) as cnt from {new_table}""".format(new_table = SEMANTIC_ANNOTATIONS_TABLE)
r2 = pd.read_sql_query(query, con)
assert r2["cnt"][0] == 31818, "semantilise rolli tabelis ei ole eelduslik arv ridu"

display_db_table(con, SEMANTIC_ANNOTATIONS_TABLE)

,pattern_id,phrase_nr,semantic_role,certainty
0,1,1,isik,vahel
1,1,1,koht,vahel
2,1,1,muu,mitte kunagi
3,2,1,isik,vahel
4,2,1,koht,vahel


In [9]:
con.close()